In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# import seaborn as sns
import os
import sys
import requests
import torch
from torch import nn, distributions
from scipy.spatial.distance import cdist
from scipy.stats import norm, multivariate_normal
import GPyOpt
import subprocess
import json

In [ ]:
result = subprocess.run(
    [r"C:\Users\karlj\anaconda3\envs\gpyopt_env\python.exe", "gpyopt_worker.py"],
    input=json.dumps({"bounds": bounds, "init_points": init_points}),
    capture_output=True,
    text=True
)
optimized_result = json.loads(result.stdout)
print(optimized_result)

In [ ]:
def get_elevations_batch(coordinates):
    """
    coordinates: list of tuples [(lat1, lon1), (lat2, lon2), ...]
    Returns: list of elevations
    """
    url = "https://maps.googleapis.com/maps/api/elevation/json"
    
    # Join coordinates with pipe separator
    locations_string = "|".join([f"{lat},{lon}" for lat, lon in coordinates])
    
    params = {
        "locations": locations_string,
        "key": API_KEY
    }
    
    response = requests.get(url, params=params)
    data = response.json()
    
    if data["status"] == "OK":
        return [result["elevation"] for result in data["results"]]
    else:
        raise Exception(f"API Error: {data['status']}")
    

# Example batch usage
coordinates = [(40.748817, -73.985428), (34.052235, -118.243683), (51.507351, -0.127758)]
elevations = get_elevations_batch(coordinates)
print("Elevations:", elevations)

In [ ]:
normal = multivariate_normal(mean=[0.3,-0.5], cov=[[2,.1],[.1,2]])
f= lambda x: -normal.pdf(x)
N = 100
max_iter = 16

longitude = (9,17)
lattitude = (46,49)
x = np.linspace(longitude[0], longitude[1], N)
y = np.linspace(lattitude[0], lattitude[1], N)
x = np.array(np.meshgrid(x,y,indexing='ij')).reshape(2,-1)

print(x.shape)

Xinit = np.array([[longitude[0], lattitude[0]], [longitude[1], lattitude[0]], [longitude[0], lattitude[1]], [longitude[1], lattitude[1]]])

bounds2d = [{'name': 'x', 'type': 'continuous', 'domain': longitude},      # longitude
            {'name': 'y', 'type': 'continuous', 'domain': lattitude}]     # lattitude

bo = GPyOpt.methods.BayesianOptimization(f, domain=bounds2d, X=Xinit, acquisition_type='EI')
bo.run_optimization(max_iter = max_iter)

# %%
y=f(x.T).reshape(N,N).T
plt.figure('Optimization progress')
im = plt.imshow(y,cmap='viridis',origin='lower',extent=[x[0][0],x[0][-1],x[1][0],x[1][-1]])
plt.plot(bo.X[:,0],bo.X[:,1],'ro:')
for i,xx in enumerate(bo.X):
    plt.text(xx[0],xx[1],'%i'%(i+1))
plt.colorbar(im)
plt.figure('Approximated function')
plt.subplot(1,2,1)
plt.title('mean')
plt.imshow(bo.model.predict(x.T)[0].reshape(N,N).T,cmap='viridis',origin='lower',extent=[x[0][0],x[0][-1],x[1][0],x[1][-1]])
plt.subplot(1,2,2)
plt.title('variance')
plt.imshow(bo.model.predict(x.T)[1].reshape(N,N).T,cmap='viridis',origin='lower',extent=[x[0][0],x[0][-1],x[1][0],x[1][-1]])
#plt.suptitle('Approximated function')
plt.figure('Acquisition function')
plt.imshow(-bo.acquisition.acquisition_function(x.T).reshape(N,N).T,cmap='viridis',origin='lower',extent=[x[0][0],x[0][-1],x[1][0],x[1][-1]])

# Bayesian Optimization with GPyOpt (via subprocess)
This cell runs GPyOpt in a separate Python 3.11 environment while querying elevation data from the current Python 3.12 environment.

In [ ]:
import subprocess
import json
import numpy as np
import matplotlib.pyplot as plt

# Define optimization parameters
longitude = (9, 17)
latitude = (46, 49)

# Initial points (corners of the region)
X_init = [
    [longitude[0], latitude[0]], 
    [longitude[1], latitude[0]], 
    [longitude[0], latitude[1]], 
    [longitude[1], latitude[1]]
]

# Bounds for optimization
bounds = [
    {'name': 'longitude', 'type': 'continuous', 'domain': longitude},
    {'name': 'latitude', 'type': 'continuous', 'domain': latitude}
]

# Optimization parameters
params = {
    "bounds": bounds,
    "X_init": X_init,
    "max_iter": 16,
    "acquisition_type": "EI",
    "vrt_path": "C:/Users/karlj/Desktop/nasadem_all/nasadem.vrt",
    "maximize": True  # Set to True to find maximum elevation, False for minimum
}

# Call GPyOpt worker in Python 3.11 environment
print("Running Bayesian Optimization with GPyOpt...")
result = subprocess.run(
    [r"C:\Users\karlj\anaconda3\envs\gpyopt_env\python.exe", "gpyopt_worker.py"],
    input=json.dumps(params),
    capture_output=True,
    text=True,
    timeout=300  # 5 minute timeout
)

if result.returncode != 0:
    print("Error:", result.stderr)
else:
    # Parse results
    opt_result = json.loads(result.stdout)
    
    print(f"\nOptimization complete!")
    print(f"Optimal location: lon={opt_result['x_opt'][0]:.4f}, lat={opt_result['x_opt'][1]:.4f}")
    print(f"Optimal elevation: {opt_result['fx_opt']:.2f} meters")
    
    # Convert to numpy arrays for plotting
    X = np.array(opt_result['X'])
    Y = np.array(opt_result['Y']).flatten()
    
    # Plot optimization progress
    plt.figure(figsize=(10, 6))
    plt.scatter(X[:, 0], X[:, 1], c=Y, cmap='viridis', s=100, edgecolors='black')
    plt.plot(X[:, 0], X[:, 1], 'r:')
    for i, (x, y) in enumerate(X):
        plt.text(x, y, str(i+1), ha='center', va='center', color='white', fontweight='bold')
    plt.colorbar(label='Elevation (m)')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.title('Bayesian Optimization Progress')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()